# Modelo: autoencoder Conv1D sobre C1

Este notebook documenta la etapa de modelo. Toma los latidos de C1 (`data/processed/beats.npz`), entrena el autoencoder con los latidos normales limpios de train (o carga un entrenamiento ya hecho) y calcula el error de reconstrucción de cada latido de train y val: el **contrato C2** que usa Evaluación (`docs/contracts.md`).

**Cómo leerlo**
- Mismo formato que el notebook de datos: cada etapa empieza diciendo qué entra, qué sale y qué decisiones aplica. Cada subsección tiene un título numerado, una explicación corta antes del código y una celda **Lectura** después del resultado.
- El código definitivo vive en `src/ecg/` (`models/autoencoder.py`, `models/preprocessing.py`, `train.py`, `score.py`) y tiene tests. Este notebook lo importa: explica, verifica y muestra.
- Los hiperparámetros salen solo de `configs/train/baseline.yaml`.
- Las líneas `OK` / `DIF` las imprime `check()`, igual que en el notebook de datos.

| Etapa | Pregunta que responde | Deja definido |
|---|---|---|
| Preparación | ¿Con qué configuración trabajamos? | `cfg`, `check()` |
| 0 · Los latidos de entrenamiento | ¿Con qué latidos aprende el modelo? | `x_train`, `x_val` |
| 1 · La arquitectura | ¿Cómo es el modelo por dentro? | `model` |
| 2 · Entrenamiento | ¿Cómo aprende y cuándo se detiene? | `run_dir` |
| 3 · Reconstrucciones | ¿Qué tan bien reconstruye cada tipo de latido? | `val_error` |
| 4 · Puntajes (C2) | ¿Qué recibe Evaluación? | `scores_train.npz`, `scores_val.npz` |
| Decisiones | ¿Qué se decidió y con qué evidencia? | — |

**Reglas que respeta.** El split de test no se carga: `load_beats` lo bloquea y `score.py` solo puntúa train y val. Este notebook no elige el umbral; eso lo hace Evaluación con C2 (D-010).

## Preparación

Importaciones, la configuración del entrenamiento, la función `check()` y una comprobación de que C1 está disponible.

### Importaciones y estilo de las gráficas

Mismos colores que en el notebook de datos: azul para el latido de entrada y naranja para la reconstrucción.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import roc_auc_score

from ecg import config
from ecg.io import load_beats
from ecg.models.autoencoder import count_parameters
from ecg.models.preprocessing import normal_mask, prepare_inputs
from ecg.paths import CONFIGS_DIR, PROCESSED_DIR, RUNS_DIR
from ecg.score import load_run, reconstruct, save_scores, score_split, validate_scores
from ecg.train import build_model, load_config, train_autoencoder
from ecg.utils.seed import seed_everything

C_MAIN = "#2a78d6"  # latido de entrada / train
C_SECOND = "#eb6834"  # reconstrucción / val
INK, MUTED = "#0b0b0b", "#52514e"

plt.rcParams.update({
    "figure.figsize": (11, 3.2),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 11,
    "lines.linewidth": 1.0,
})
pd.set_option("display.width", 120)

### Configuración y run

`cfg` es el YAML del baseline. `RUN_ID = None` usa el último entrenamiento hecho con esta configuración y, si no hay ninguno, entrena uno nuevo (entre 30 y 45 minutos en CPU). Para cargar otro entrenamiento, pon aquí su nombre de carpeta en `runs/`.

In [ ]:
CONFIG_PATH = CONFIGS_DIR / "train" / "baseline.yaml"
AUTHOR = "samugarciar"  # autor del run_id si hay que entrenar
RUN_ID = None  # None: el último run de esta configuración, o uno nuevo si no hay

cfg = load_config(CONFIG_PATH)
FS, PRE = config.FS, config.WIN_PRE
T_MS = (np.arange(cfg["model"]["input_length"]) - PRE) * 1000 / FS  # ms desde la anotación
print(json.dumps(cfg, indent=2, ensure_ascii=False))

### `check()`: comparar sin corregir

La misma función del notebook de datos: imprime `OK` o `DIF` y guarda cada diferencia en `DISCREPANCIES`, que se resume al final. Nada se ajusta para que cuadre.

In [ ]:
DISCREPANCIES: list[dict] = []


def check(name, got, expected):
    """Compara `got` con `expected`; si difieren, registra la discrepancia (nunca la corrige)."""
    ok = bool(got == expected)
    print(f"{'OK ' if ok else 'DIF'} · {name}: {got}" + ("" if ok else f"  (esperado: {expected})"))
    if not ok:
        DISCREPANCIES.append({"check": name, "obtenido": got, "esperado": expected})
    return ok

### C1 está disponible

El modelo se entrena con la versión 1.0.0 de C1. Si `beats.npz` no existe, hay que correr antes `notebooks/datos_mitbih_c1.ipynb`.

In [ ]:
c1_meta = json.loads((PROCESSED_DIR / "meta.json").read_text(encoding="utf-8"))
check("versión de C1", c1_meta["data_version"], "1.0.0")

## Etapa 0 · Los latidos de entrenamiento

El autoencoder solo debe ver latidos normales y limpios: así aprende únicamente cómo es un latido normal.

- **Entra:** C1, splits train y val.
- **Sale:** `x_train` y `x_val`, los latidos normalizados con los que se entrena y se decide cuándo parar.
- **Decisiones:** D-005 (normal = símbolo N), D-020 (sin ruidosos) y D-008 (z-score por latido).

### 0.1 Cargar C1

`load_beats()` devuelve train y val y bloquea test.

In [ ]:
beats = load_beats()
n_split = pd.Series(beats["split"]).value_counts()
check("latidos de train en C1", int(n_split["train"]), 40_996)
check("latidos de val en C1", int(n_split["val"]), 10_006)

### 0.2 Qué latidos se usan (D-020)

De cada split se toman los latidos con `label == 0` (símbolo N) y se quitan los que tienen `mlii_noisy`. Los de train enseñan al modelo; los de val deciden cuándo detener el entrenamiento.

In [ ]:
rows = []
for split in ("train", "val"):
    all_n = normal_mask(beats, split, exclude_noisy=False)
    clean_n = normal_mask(beats, split, exclude_noisy=True)
    rows.append((split, int(all_n.sum()), int((all_n & ~clean_n).sum()), int(clean_n.sum())))
display(pd.DataFrame(rows, columns=["split", "N", "N ruidosos", "N limpios (se usan)"])
        .set_index("split"))

train_mask = normal_mask(beats, "train", cfg["data"]["exclude_noisy"])
val_mask = normal_mask(beats, "val", cfg["data"]["exclude_noisy"])
check("N limpios de train (entrenamiento)", int(train_mask.sum()), 28_893)
check("N limpios de val (detención temprana)", int(val_mask.sum()), 8_451)

### 0.3 Normalizar cada latido (D-008)

El z-score le resta a cada latido su media y lo divide por su desviación estándar. Así todos quedan en la misma escala y el modelo compara formas, no amplitudes: la amplitud depende mucho del paciente y de los electrodos. La figura compara tres pacientes de train (el de menor, el del medio y el de mayor amplitud típica) antes y después de normalizar.

In [ ]:
x_train = prepare_inputs(beats["X"][train_mask], cfg["data"]["normalization"])
x_val = prepare_inputs(beats["X"][val_mask], cfg["data"]["normalization"])
check("media por latido ≈ 0", bool(np.allclose(x_train.mean(axis=1), 0, atol=1e-4)), True)
check("desviación por latido ≈ 1", bool(np.allclose(x_train.std(axis=1), 1, atol=1e-3)), True)

amplitude = pd.Series(np.ptp(beats["X"][train_mask], axis=1))
by_record = amplitude.groupby(beats["record"][train_mask]).median().sort_values()
picked = [by_record.index[0], by_record.index[len(by_record) // 2], by_record.index[-1]]
fig, (ax_mv, ax_z) = plt.subplots(1, 2, figsize=(13, 3.4), sharex=True)
for record, color in zip(picked, (C_MAIN, C_SECOND, INK)):
    row = int(np.flatnonzero(beats["record"][train_mask] == record)[100])
    ax_mv.plot(T_MS, beats["X"][train_mask][row], color=color, label=f"registro {record}")
    ax_z.plot(T_MS, x_train[row], color=color, label=f"registro {record}")
ax_mv.set(xlabel="ms desde la anotación", ylabel="mV", title="Antes: en milivoltios")
ax_z.set(xlabel="ms desde la anotación", ylabel="z-score", title="Después: z-score por latido")
ax_z.legend(loc="upper right")
plt.show()

**Lectura.** En milivoltios, el pico R va de cerca de 1 mV (registro 201) a más de 2 mV (registro 220). Después del z-score los tres quedan en la misma escala, y lo que los distingue es la forma: por ejemplo, la onda S profunda del 220. El modelo tiene que aprender esas diferencias de forma, no las de amplitud.

## Etapa 1 · La arquitectura

Un reloj de arena: el encoder comprime cada latido a 32 números y el decoder intenta reconstruirlo a partir de ellos. Como solo ve latidos normales, esos 32 números aprenden a describir variaciones de un latido normal; un latido anómalo no se puede describir bien con ellos y su reconstrucción sale peor.

- **Entra:** la sección `model` del YAML.
- **Sale:** `model`, un `ConvAutoencoder` sin entrenar.
- **Decisiones:** D-009 y D-022.

### 1.1 Construir el modelo desde el YAML

`build_model()` lee los canales, el tamaño de filtro y el tamaño del cuello del YAML. La semilla fija los pesos iniciales.

In [ ]:
seed_everything(cfg["seed"])
model = build_model(cfg)
parts = {
    "encoder (convoluciones)": count_parameters(model.encoder),
    "densa → cuello": count_parameters(model.to_latent),
    "densa ← cuello": count_parameters(model.from_latent),
    "decoder (convoluciones)": count_parameters(model.decoder),
}
display(pd.Series(parts, name="parámetros").to_frame())
check("parámetros del modelo", count_parameters(model), 284_961)

### 1.2 Formas capa por capa

Se pasa un latido de prueba por cada bloque y se anota su forma (canales, muestras). En el encoder cada bloque reduce el largo a la mitad y aumenta los canales, que son detectores de patrones locales. El decoder recorre el camino inverso.

In [ ]:
model.eval()
h = torch.zeros(1, 1, cfg["model"]["input_length"])
rows = [("entrada", tuple(h.shape[1:]))]
with torch.no_grad():
    for i, block in enumerate(model.encoder, 1):
        h = block(h)
        rows.append((f"encoder {i}", tuple(h.shape[1:])))
    z = model.to_latent(h.flatten(1))
    rows.append(("cuello", tuple(z.shape[1:])))
    h = model.from_latent(z).view(1, model.channels[-1], model.bottleneck_length)
    rows.append(("densa de vuelta", tuple(h.shape[1:])))
    step = 0
    for layer in model.decoder:
        h = layer(h)
        if not isinstance(layer, torch.nn.Upsample):
            step += 1
            rows.append((f"decoder {step}", tuple(h.shape[1:])))
display(pd.DataFrame(rows, columns=["capa", "forma (canales, muestras)"]).set_index("capa").T)
check("tamaño del cuello", rows[5][1], (cfg["model"]["latent_dim"],))
check("la salida tiene la forma de la entrada", rows[-1][1], (1, cfg["model"]["input_length"]))

### 1.3 Qué alcanza a ver el encoder

Cada filtro mira 7 muestras (~19 ms), pero al apilar cuatro capas que reducen el largo a la mitad, cada posición del final resume un tramo mucho más largo de la señal: su *campo receptivo*.

In [ ]:
kernel, levels = cfg["model"]["kernel_size"], len(cfg["model"]["channels"])
receptive_field = 1 + (kernel - 1) * (2**levels - 1)
print(f"campo receptivo: {receptive_field} muestras = {receptive_field * 1000 / FS:.0f} ms")
check("campo receptivo del encoder (muestras)", receptive_field, 91)

**Lectura.** 91 muestras son 253 ms: alcanzan para ver un QRS completo (~100 ms) junto con parte de la onda P o de la T. La capa densa del cuello junta después todas las posiciones, así que los 32 números resumen el latido entero.

### 1.4 Antes de entrenar

Un modelo con pesos al azar todavía no sabe nada. Su reconstrucción de un latido normal de val sirve como punto de partida.

In [ ]:
inputs, outputs = reconstruct(model, beats["X"][val_mask][:1], cfg["data"]["normalization"])
untrained_mse = float(((inputs - outputs) ** 2).mean())
fig, ax = plt.subplots()
ax.plot(T_MS, inputs[0], color=C_MAIN, label="latido (z-score)")
ax.plot(T_MS, outputs[0], color=C_SECOND, label="reconstrucción sin entrenar")
ax.set(xlabel="ms desde la anotación", ylabel="z-score",
       title=f"Modelo sin entrenar · MSE {untrained_mse:.2f}")
ax.legend(loc="upper right")
plt.show()

**Lectura.** La reconstrucción es casi plana. Con latidos normalizados, un MSE cercano a 1 equivale a no reconstruir nada: es el error que da devolver siempre la media. El entrenamiento tiene que bajarlo muy por debajo de eso.

## Etapa 2 · Entrenamiento

- **Entra:** `x_train`, `x_val` y la sección `training` del YAML.
- **Sale:** `run_dir`, la carpeta `runs/<run_id>/` con los pesos de la mejor época (`model.pt`), la historia por época (`history.json`), la configuración usada y `meta.json`.
- **Decisión:** D-021 (detención temprana).

### 2.1 Cómo aprende

- **Época:** una pasada por los 28 893 latidos de train, en lotes de 256 y en un orden al azar fijado por la semilla.
- **Pérdida:** el MSE entre cada latido y su reconstrucción. El optimizador Adam ajusta los pesos para bajarlo.
- **Detención temprana (D-021):** al final de cada época se mide el MSE de los N limpios de val. Si no mejora en 10 épocas seguidas, se detiene y se conservan los pesos de la mejor época. El máximo es 100 épocas.

Es lo mismo que hace desde la terminal `python -m ecg.train --config configs/train/baseline.yaml --author <autor>`.

### 2.2 Entrenar o cargar el run

Si ya existe un run de esta configuración, se carga; si no, se entrena uno nuevo. Las comprobaciones confirman que el run corresponde a este YAML y a estos datos.

In [ ]:
def latest_run(name: str):
    """Último run terminado (con meta.json) de una configuración, o None."""
    done = sorted(p for p in RUNS_DIR.glob(f"*_{name}") if (p / "meta.json").exists())
    return done[-1] if done else None


if RUN_ID is not None:
    run_dir = RUNS_DIR / RUN_ID
else:
    run_dir = latest_run(cfg["name"]) or train_autoencoder(cfg, AUTHOR)

run_meta = json.loads((run_dir / "meta.json").read_text(encoding="utf-8"))
history = pd.DataFrame(json.loads((run_dir / "history.json").read_text(encoding="utf-8")))
summary_keys = ("run_id", "git_commit", "data_version", "n_train", "n_val", "epochs_run",
                "best_epoch", "best_val_loss", "seconds", "device")
display(pd.Series({k: run_meta[k] for k in summary_keys}).to_frame("valor"))
check("el run usa la configuración actual", load_config(run_dir / "config.yaml") == cfg, True)
check("el run entrenó con los N limpios de train", run_meta["n_train"], int(train_mask.sum()))
check("el run se detuvo con los N limpios de val", run_meta["n_val"], int(val_mask.sum()))
check("el run usó C1 1.0.0", run_meta["data_version"], "1.0.0")

### 2.3 Curvas de aprendizaje

El error de train y el de val, época por época, en escala logarítmica. La línea punteada marca la mejor época, la que se guardó.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(history["epoch"], history["train_loss"], color=C_MAIN, label="train (N limpios)")
ax.plot(history["epoch"], history["val_loss"], color=C_SECOND, label="val (N limpios)")
ax.axvline(run_meta["best_epoch"], color=MUTED, ls="--", lw=0.8)
ax.set_yscale("log")
ax.set(xlabel="época", ylabel="MSE (z-score)",
       title=f"Curvas de aprendizaje · mejor época {run_meta['best_epoch']}")
ax.legend(loc="upper right")
plt.show()

best_row = history.loc[history["val_loss"].idxmin()]
check("la mejor época es la de menor error en val", int(best_row["epoch"]), run_meta["best_epoch"])
print(f"en la mejor época: train {best_row['train_loss']:.5f} · val {best_row['val_loss']:.5f}")

**Lectura.** Se completa con los resultados del primer entrenamiento del baseline.

## Etapa 3 · Reconstrucciones

- **Entra:** el modelo con los pesos de la mejor época y todos los latidos de val, normales y anómalos.
- **Sale:** `val_error`, el error de reconstrucción de cada latido de val.

Para no elegir ejemplos a dedo, cada figura muestra el latido con el **error mediano** de su grupo: ni el mejor ni el peor caso.

### 3.1 Cargar el modelo entrenado y reconstruir val

In [ ]:
model, _ = load_run(run_dir)
val_rows = np.flatnonzero(beats["split"] == "val")
val_inputs, val_outputs = reconstruct(model, beats["X"][val_rows], cfg["data"]["normalization"])
val_error = ((val_inputs - val_outputs) ** 2).mean(axis=1)
val_info = pd.DataFrame({
    "record": beats["record"][val_rows], "symbol": beats["symbol"][val_rows],
    "aami": beats["aami"][val_rows], "noisy": beats["mlii_noisy"][val_rows], "error": val_error,
})


def median_example(mask) -> int:
    """Posición (en val) del latido con el error mediano dentro de `mask`."""
    positions = np.flatnonzero(mask)
    return int(positions[np.argsort(val_error[positions])][len(positions) // 2])


def plot_examples(positions, titles) -> None:
    fig, axes = plt.subplots(1, len(positions), figsize=(13, 2.9), sharey=True)
    for ax, pos, title in zip(axes, positions, titles):
        ax.plot(T_MS, val_inputs[pos], color=C_MAIN, label="latido")
        ax.plot(T_MS, val_outputs[pos], color=C_SECOND, label="reconstrucción")
        ax.set_title(f"{title} · MSE {val_error[pos]:.3f}", fontsize=9)
        ax.set_xlabel("ms desde la anotación")
    axes[0].set_ylabel("z-score")
    axes[0].legend(loc="upper left", fontsize=8)
    plt.show()


print(f"{len(val_rows):,} latidos de val reconstruidos")

### 3.2 Latidos normales

Un latido N limpio por cada paciente de val.

In [ ]:
normal_clean = (val_info["symbol"] == "N") & ~val_info["noisy"]
plot_examples([median_example(normal_clean & (val_info["record"] == r)) for r in config.DS1_VAL],
              [f"N · registro {r}" for r in config.DS1_VAL])

**Lectura.** Se completa con los resultados del primer entrenamiento del baseline.

### 3.3 Latidos anómalos

Los símbolos distintos de N con al menos 10 latidos en val.

In [ ]:
counts = val_info.loc[val_info["symbol"] != "N", "symbol"].value_counts()
anomalous = [s for s, n in counts.items() if n >= 10][:4]
plot_examples([median_example(val_info["symbol"] == s) for s in anomalous],
              [f"{s} (AAMI {config.AAMI_MAP[s]}) · n = {counts[s]}" for s in anomalous])

**Lectura.** Se completa con los resultados del primer entrenamiento del baseline.

## Etapa 4 · Puntajes para Evaluación (contrato C2)

- **Entra:** el run y C1.
- **Sale:** `runs/<run_id>/scores_train.npz` y `scores_val.npz`, con el error de cada latido del split (también los ruidosos) y sus metadatos.

### 4.1 Calcular y guardar C2

`score_split()` usa la misma normalización del entrenamiento, valida el contrato y guarda en `idx` la fila de cada latido dentro de `beats.npz`. Es lo mismo que `python -m ecg.score --run runs/<run_id> --split val`.

In [ ]:
scores = {}
for split in ("train", "val"):
    scores[split] = score_split(run_dir, split)
    path = save_scores(run_dir, split, scores[split])
    print(f"{path.name}: {len(scores[split]['idx']):,} latidos")
check("C2 de train cubre todos sus latidos", len(scores["train"]["idx"]), 40_996)
check("C2 de val cubre todos sus latidos", len(scores["val"]["idx"]), 10_006)
check("C2 de val cumple el contrato", validate_scores(scores["val"]), True)

### 4.2 Consistencia con el entrenamiento

Dos comprobaciones de que C2 mide lo mismo que se optimizó: el error medio de los N limpios de val tiene que ser el de la mejor época, y los errores de C2 tienen que coincidir con los de la etapa 3.

In [ ]:
val_scores = scores["val"]
clean_normal = (val_scores["label"] == 0) & ~val_scores["mlii_noisy"]
check("error medio de los N limpios de val = el de la mejor época",
      bool(np.isclose(val_scores["error"][clean_normal].mean(), run_meta["best_val_loss"],
                      rtol=1e-4)), True)
check("C2 de val = errores de la etapa 3",
      bool(np.allclose(val_scores["error"], val_error, rtol=1e-5)), True)

### 4.3 Error por clase en val

La distribución del error por clase AAMI (escala logarítmica). La línea punteada es el percentil 95 de los N limpios de val, solo como referencia: el umbral oficial lo calcula Evaluación (D-010).

In [ ]:
error_df = pd.DataFrame({k: val_scores[k] for k in ("error", "aami", "symbol", "label")})
classes = [c for c in ["N", "S", "V", "F", "Q"] if (error_df["aami"] == c).any()]
reference_p95 = float(np.quantile(val_scores["error"][clean_normal], 0.95))

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.boxplot([error_df.loc[error_df["aami"] == c, "error"] for c in classes], showfliers=False)
ax.set_xticks(range(1, len(classes) + 1), classes)
ax.axhline(reference_p95, color=MUTED, ls="--", lw=0.8)
ax.set_yscale("log")
ax.set(xlabel="clase AAMI", ylabel="error de reconstrucción (MSE)",
       title="Val · error por clase (cajas sin valores extremos)")
plt.show()

by_symbol = error_df.groupby("symbol")["error"].describe(percentiles=[0.5])[["count", "50%"]]
by_symbol["sobre la referencia"] = error_df.groupby("symbol")["error"].apply(
    lambda e: f"{(e > reference_p95).mean():.0%}")
display(by_symbol.rename(columns={"count": "latidos", "50%": "error mediano"})
        .sort_values("error mediano"))
print(f"referencia (P95 de los N limpios de val): {reference_p95:.4f}")

**Lectura.** Se completa con los resultados del primer entrenamiento del baseline.

### 4.4 Diagnóstico rápido

Un único número para saber si el modelo separa normales de anómalos en val: la AUC-ROC. Va de 0,5 (azar) a 1 (separación perfecta) y no depende de ningún umbral. No es el reporte oficial, que es C3 y lo produce Evaluación.

In [ ]:
is_anomaly = val_scores["label"] == 1
auc = {"todas las anomalías": roc_auc_score(is_anomaly, val_scores["error"])}
for cls in ["N", "S", "V", "F", "Q"]:
    positives = is_anomaly & (val_scores["aami"] == cls)
    if positives.sum() >= 10:
        subset = ~is_anomaly | positives
        name = "N no estricto (e, j…)" if cls == "N" else cls
        auc[f"{name} contra N"] = roc_auc_score(positives[subset], val_scores["error"][subset])
display(pd.Series(auc, name="AUC-ROC en val").round(3).to_frame())

**Lectura.** Se completa con los resultados del primer entrenamiento del baseline.

## Decisiones del modelo y discrepancias

| | Decisión | Dónde se ve |
|---|---|---|
| D-008 | z-score por latido antes del modelo | 0.3 |
| D-009 · D-022 | Autoencoder Conv1D de cuatro niveles con cuello denso de 32 | Etapa 1 |
| D-020 | Latidos ruidosos fuera del entrenamiento y de la detención temprana | 0.2 |
| D-021 | Detención temprana por el error de los N limpios de val | 2.3 |
| D-010 | Umbral = percentil 95 de los N de val; lo calcula Evaluación con C2 | 4.3 |

La última celda resume todo lo que no coincidió con lo esperado.

In [ ]:
if DISCREPANCIES:
    display(pd.DataFrame(DISCREPANCIES))
else:
    print("Sin discrepancias.")